In [2]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [4]:
PRODUCTS = {
        "wireless headphones": {"price": 79.99, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
        "smart watch":         {"price": 199.99, "description": "Tracks heart rate and sleep. 5-day battery, water-resistent."},
        "mechanical keyboard": {"price": 129.00, "description": "Tenkeyless, Cherry MX Brown switces, per--key RGB."},
        "laptop stand":        {"price": 34.99, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}    

@tool 
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, and description."""
    p =PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {',' .join(PRODUCTS)}"
    return str(p)   

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent = create_agent(
      llm,
     tools=[get_product],
     system_prompt ="You are a helpful product assistant for an online tech store.",

 )     

In [5]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [6]:
ask("what is the price of wireless headphone.")

The price of the wireless headphones is $79.99. They are over-ear Bluetooth headphones with a 30-hour battery life and active noise cancellation.


In [8]:
REVIEWS = { 
    "wireless headphones": {"riviews": 1262,  "rating": 4.6},
    "smart watch":         {"riviews": 340,   "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,    "rating": 4.8},
    "laptop stand":        {"reviews": 781,   "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name: Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)
    

In [9]:
llm = ChatGroq(model= "llama-3.3-70b-versatile", temperature= 0)

agent2 =create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistent for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)
    

In [10]:
ask2("do people like smart watches? do not include information about the product")

People generally like smart watches, with an average rating of 3.9 out of 5 based on 340 reviews.


In [11]:
ask2("how much does each cost?")

I don't have enough information to provide the prices. Please provide the product names.


In [12]:
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask2(question: str):
    config = {"configurable": {"thread_id": "user-ben_session-1"}}
    result = agent2.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config
    )
    print(result["messages"][-1].content)

In [13]:
ask2("What is the price of the watch?")

The price of the smart watch is $199.99. It tracks heart rate and sleep, has a 5-day battery, and is water-resistant.


In [14]:
ask2("does it have good reviews?")

The smart watch has 340 reviews with an average rating of 3.9, indicating generally positive reviews.


In [15]:
ask2("What are the benefits of buying it?")

The benefits of buying the smart watch include:

1. Tracking heart rate and sleep patterns to monitor overall health and wellness.
2. Having a 5-day battery life, making it convenient for daily use without needing frequent recharging.
3. Being water-resistant, which provides protection against accidental splashes or exposure to water.

These features can be beneficial for individuals who want to stay on top of their fitness and health, as well as those who lead active lifestyles and need a reliable accessory.


In [16]:
ask2("Thank you for your information")

You're welcome. It was my pleasure to assist you with information about the smart watch. If you have any more questions or need further assistance, feel free to ask.
